# Análisis de Audio con Basic-Pitch (Spotify)
Extrae notas, pitch y genera un archivo MIDI a partir de cualquier grabación de audio.  
Ideal para analizar quenas, pincuyos y traversas.

In [71]:
# Verificar que basic-pitch está instalado correctamente
import importlib.metadata
try:
    version = importlib.metadata.version("basic-pitch")
    print(f"✓ basic-pitch versión: {version}")
except importlib.metadata.PackageNotFoundError:
    print("⚠ basic-pitch no está instalado en este kernel.")
    print("  Ejecuta: pip install basic-pitch")


✓ basic-pitch versión: 0.4.0


## 1. Importaciones

In [72]:
import os
import numpy as np
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH
import pretty_midi

## 2. Configuración del archivo de audio
Cambia `AUDIO_FILE` por la ruta a tu grabación (`.wav`, `.mp3`, `.ogg`, `.flac`).

In [73]:
# =====================================================================
# CONFIGURA AQUÍ TU ARCHIVO DE AUDIO
# =====================================================================
AUDIO_FILE = "7. Juni um 21-51.m4a"   # grabación adjunta

# Parámetros de detección (ajustables)
ONSET_THRESHOLD = 1    # sensibilidad para detectar inicio de nota (0-1)
FRAME_THRESHOLD = 0.195   # sensibilidad por frame (0-1)
MIN_NOTE_LEN    = 127    # duración mínima de nota en ms
MIN_FREQ        = 200.0  # frecuencia mínima en Hz (filtra ruido bajo)
MAX_FREQ        = 2000.0 # frecuencia máxima en Hz

# Verificar que el archivo existe
if not os.path.exists(AUDIO_FILE):
    print(f"⚠ Archivo '{AUDIO_FILE}' no encontrado.")
    print("  Coloca tu grabación en la carpeta del proyecto y actualiza AUDIO_FILE.")
else:
    print(f"✓ Archivo encontrado: {AUDIO_FILE}")


✓ Archivo encontrado: 7. Juni um 21-51.m4a


In [74]:
import subprocess
import imageio_ffmpeg

FFMPEG_EXE = imageio_ffmpeg.get_ffmpeg_exe()
print(f"ffmpeg: {FFMPEG_EXE}")

# Convertir m4a → wav si es necesario
SRC_FILE = "7. Juni um 21-51.m4a"
WAV_FILE  = "7. Juni um 21-51.wav"

if not os.path.exists(WAV_FILE):
    print(f"Convirtiendo {SRC_FILE} → {WAV_FILE} ...")
    result = subprocess.run(
        [FFMPEG_EXE, "-y", "-i", SRC_FILE, "-ar", "22050", "-ac", "1", WAV_FILE],
        capture_output=True, text=True, encoding="utf-8", errors="replace"
    )
    if result.returncode != 0:
        print("Error:", result.stderr[-600:])
    else:
        print(f"✓ Convertido: {WAV_FILE}")
else:
    print(f"✓ WAV ya existe: {WAV_FILE}")

AUDIO_FILE = WAV_FILE
print(f"Usando: {AUDIO_FILE}")


ffmpeg: c:\Users\CESAR FERNANDO\.conda\envs\Musik_Allgemain\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe
✓ WAV ya existe: 7. Juni um 21-51.wav
Usando: 7. Juni um 21-51.wav


## 3. Análisis con Basic-Pitch

In [75]:
if os.path.exists(AUDIO_FILE):
    print("Analizando audio... (puede tardar unos segundos)\n")

    model_output, midi_data, note_events = predict(
        AUDIO_FILE,
        onset_threshold=ONSET_THRESHOLD,
        frame_threshold=FRAME_THRESHOLD,
        minimum_note_length=MIN_NOTE_LEN,
        minimum_frequency=MIN_FREQ,
        maximum_frequency=MAX_FREQ,
    )

    print(f"✓ Análisis completado. Notas detectadas: {len(note_events)}\n")
    print(f"{'#':<5} {'Inicio(s)':<12} {'Fin(s)':<10} {'MIDI':<8} {'Freq(Hz)':<12} {'Vel.':<6}")
    print("-" * 55)
    for i, (start, end, pitch, vel, _) in enumerate(note_events):
        freq = 440.0 * (2.0 ** ((pitch - 69) / 12.0))
        print(f"{i+1:<5} {start:<12.3f} {end:<10.3f} {pitch:<8} {freq:<12.1f} {vel:.2f}")
else:
    print("⚠ Saltando análisis: archivo de audio no disponible.")
    note_events = []
    midi_data = None

Analizando audio... (puede tardar unos segundos)

Predicting MIDI for 7. Juni um 21-51.wav...


✓ Análisis completado. Notas detectadas: 104

#     Inicio(s)    Fin(s)     MIDI     Freq(Hz)     Vel.  
-------------------------------------------------------
1     7.156        7.423      93       1760.0       0.56
2     7.423        7.958      90       1480.0       0.57
3     14.684       14.951     93       1760.0       0.54
4     10.978       11.291     93       1760.0       0.54
5     32.970       33.295     91       1568.0       0.53
6     34.805       35.119     91       1568.0       0.52
7     31.413       31.925     88       1318.5       0.36
8     14.928       15.532     90       1480.0       0.58
9     9.305        9.804      88       1318.5       0.62
10    17.170       17.425     88       1318.5       0.52
11    28.253       28.741     91       1568.0       0.44
12    28.880       29.182     91       1568.0       0.56
13    39.394       40.788     88       1318.5       0.52
14    13.673       13.953     88       1318.5       0.53
15    21.015       21.201     88       13

## 4. Convertir notas MIDI a nombres musicales

In [76]:
NOTAS_ES = ["Do", "Do#", "Re", "Re#", "Mi", "Fa",
            "Fa#", "Sol", "Sol#", "La", "La#", "Si"]

def midi_a_nombre(midi_num):
    nombre = NOTAS_ES[midi_num % 12]
    octava = (midi_num // 12) - 1
    return f"{nombre}{octava}"

if note_events:
    print(f"\n{'#':<5} {'Nota':<8} {'Inicio(s)':<12} {'Duración(ms)':<15} {'Freq(Hz)':<12}")
    print("-" * 55)
    for i, (start, end, pitch, vel, _) in enumerate(note_events):
        freq   = 440.0 * (2.0 ** ((pitch - 69) / 12.0))
        dur_ms = (end - start) * 1000
        print(f"{i+1:<5} {midi_a_nombre(int(pitch)):<8} {start:<12.3f} {dur_ms:<15.0f} {freq:<12.1f}")
else:
    print("Sin notas detectadas.")


#     Nota     Inicio(s)    Duración(ms)    Freq(Hz)    
-------------------------------------------------------
1     La6      7.156        267             1760.0      
2     Fa#6     7.423        535             1480.0      
3     La6      14.684       267             1760.0      
4     La6      10.978       313             1760.0      
5     Sol6     32.970       325             1568.0      
6     Sol6     34.805       313             1568.0      
7     Mi6      31.413       512             1318.5      
8     Fa#6     14.928       604             1480.0      
9     Mi6      9.305        499             1318.5      
10    Mi6      17.170       255             1318.5      
11    Sol6     28.253       488             1568.0      
12    Sol6     28.880       302             1568.0      
13    Mi6      39.394       1394            1318.5      
14    Mi6      13.673       280             1318.5      
15    Mi6      21.015       186             1318.5      
16    Si5      32.482       302

## 5. Guardar el resultado como archivo MIDI

In [77]:
if midi_data is not None:
    OUTPUT_MIDI = AUDIO_FILE.rsplit(".", 1)[0] + "_notas.mid"
    midi_data.write(OUTPUT_MIDI)
    print(f"✓ MIDI guardado en: {OUTPUT_MIDI}")
    print("  Puedes abrirlo en MuseScore, GarageBand o importarlo con music21.")
else:
    print("Sin datos MIDI para guardar.")

✓ MIDI guardado en: 7. Juni um 21-51_notas.mid
  Puedes abrirlo en MuseScore, GarageBand o importarlo con music21.


## 6. Estadísticas de la grabación

In [78]:
if note_events:
    from collections import Counter

    pitches   = [int(n[2]) for n in note_events]
    freqs     = [440.0 * (2.0 ** ((p - 69) / 12.0)) for p in pitches]
    durations = [(n[1] - n[0]) * 1000 for n in note_events]
    conteo    = Counter([midi_a_nombre(p) for p in pitches])

    print("=== ESTADÍSTICAS ===")
    print(f"  Total de notas        : {len(note_events)}")
    print(f"  Nota más grave        : {midi_a_nombre(min(pitches))}  ({min(freqs):.1f} Hz)")
    print(f"  Nota más aguda        : {midi_a_nombre(max(pitches))}  ({max(freqs):.1f} Hz)")
    print(f"  Duración media (ms)   : {np.mean(durations):.0f}")
    print(f"  Duración mínima (ms)  : {np.min(durations):.0f}")
    print(f"  Duración máxima (ms)  : {np.max(durations):.0f}")
    print(f"\n  Notas más frecuentes:")
    for nota, cnt in conteo.most_common(8):
        barra = "█" * cnt
        print(f"    {nota:<6} {barra}  ({cnt})")
else:
    print("Sin datos para estadísticas.")

=== ESTADÍSTICAS ===
  Total de notas        : 104
  Nota más grave        : Re5  (587.3 Hz)
  Nota más aguda        : La#6  (1864.7 Hz)
  Duración media (ms)   : 279
  Duración mínima (ms)  : 139
  Duración máxima (ms)  : 1394

  Notas más frecuentes:
    La6    ████████████████████████  (24)
    Sol6   ███████████████████████  (23)
    Mi6    ██████████████████████  (22)
    Si5    ████████████████████  (20)
    Fa#6   ███████  (7)
    Re6    ███  (3)
    La5    ███  (3)
    Re5    █  (1)


## 7. Exportar lista de notas a CSV

In [79]:
import csv

if note_events:
    OUTPUT_CSV = AUDIO_FILE.rsplit(".", 1)[0] + "_notas.csv"
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["#", "Nota", "MIDI", "Freq_Hz",
                         "Inicio_s", "Fin_s", "Duracion_ms", "Velocidad"])
        for i, (start, end, pitch, vel, _) in enumerate(note_events):
            freq   = 440.0 * (2.0 ** ((pitch - 69) / 12.0))
            dur_ms = (end - start) * 1000
            writer.writerow([i+1, midi_a_nombre(int(pitch)), int(pitch),
                             round(freq, 2), round(start, 3), round(end, 3),
                             round(dur_ms, 0), round(vel, 3)])
    print(f"✓ CSV guardado en: {OUTPUT_CSV}")
else:
    print("Sin notas para exportar.")

✓ CSV guardado en: 7. Juni um 21-51_notas.csv


## 7. Reproducir las notas detectadas

Síntesis de las notas con armónicos (flautín) y reproductor integrado en el notebook.

In [80]:
from IPython.display import HTML, display
import base64, io, wave, json, re
import numpy as np

SR = 22050  # Hz

# ── Digitaciones del pincuyo ────────────────────────────────────────────────
# 6 orificios frontales, SIN orificio trasero (pulgar)
# Orden: [h1, h2, h3, h4, h5, h6]  — h1 = más cercano a la boca
# 1 = tapado  |  0 = abierto
DIGITACIONES = {
    "Re":   [1, 1, 1, 1, 1, 1],
    "Mi":   [1, 1, 1, 1, 1, 0],
    "Fa#":  [1, 1, 1, 1, 0, 0],
    "Sol":  [1, 1, 1, 0, 0, 0],
    "La":   [1, 1, 0, 0, 0, 0],
    "Si":   [1, 0, 0, 0, 0, 0],
    "Do#":  [0, 0, 0, 0, 0, 0],
    "Re#":  [1, 1, 1, 1, 0, 1],
    "Fa":   [1, 1, 1, 0, 1, 1],
    "Sol#": [1, 1, 0, 1, 0, 0],
    "La#":  [1, 0, 1, 0, 0, 0],
    "Do":   [0, 1, 0, 0, 0, 0],
    "Si#":  [0, 0, 0, 0, 0, 0],
}
DEFAULT_DIG = [1, 1, 1, 1, 1, 1]

def nota_base(nombre):
    return re.sub(r'-?\d+$', '', str(nombre))

def sintetizar_nota(freq, duracion_s, vel=0.5, sr=SR, fade=0.015):
    t = np.linspace(0, duracion_s, int(sr * duracion_s), endpoint=False)
    senal = (0.65 * np.sin(2 * np.pi * freq * t)
           + 0.25 * np.sin(2 * np.pi * 2 * freq * t)
           + 0.10 * np.sin(2 * np.pi * 3 * freq * t))
    senal *= float(vel)
    n_fade = int(fade * sr)
    if len(senal) > 2 * n_fade:
        senal[:n_fade]  *= np.linspace(0.0, 1.0, n_fade)
        senal[-n_fade:] *= np.linspace(1.0, 0.0, n_fade)
    return senal.astype(np.float32)

if note_events:
    total_dur = max(end for _, end, _, _, _ in note_events) + 0.5
    audio_out  = np.zeros(int(SR * total_dur), dtype=np.float32)

    for start, end, pitch, vel, _ in note_events:
        freq     = 440.0 * (2.0 ** ((pitch - 69) / 12.0))
        duracion = max(end - start, 0.05)
        senal    = sintetizar_nota(freq, duracion, vel=float(vel))
        i0 = int(start * SR)
        i1 = i0 + len(senal)
        if i1 <= len(audio_out):
            audio_out[i0:i1] += senal

    mx = np.max(np.abs(audio_out))
    if mx > 0:
        audio_out /= mx

    buf = io.BytesIO()
    with wave.open(buf, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(SR)
        wf.writeframes((audio_out * 32767).astype(np.int16).tobytes())
    audio_b64 = base64.b64encode(buf.getvalue()).decode()

    notes_js = [
        [float(start), float(end), int(pitch),
         midi_a_nombre(int(pitch)),
         round(440.0 * (2.0 ** ((int(pitch) - 69) / 12.0)), 1),
         DIGITACIONES.get(nota_base(midi_a_nombre(int(pitch))), DEFAULT_DIG)]
        for start, end, pitch, vel, _ in note_events
    ]

    pills_html = "".join(
        f'<span class="nota-pill" id="pill-{i}">{n[3]}</span>'
        for i, n in enumerate(notes_js)
    )

    # SVG del pincuyo:  tapado = azul  |  abierto = negro
    pincuyo_svg = """
<svg id="pincuyo-svg" width="72" height="340" viewBox="0 0 72 340" xmlns="http://www.w3.org/2000/svg">
  <defs>
    <linearGradient id="body-grad" x1="0%" y1="0%" x2="100%" y2="0%">
      <stop offset="0%"   stop-color="#5C3A0A"/>
      <stop offset="35%"  stop-color="#9B7220"/>
      <stop offset="65%"  stop-color="#C89A30"/>
      <stop offset="100%" stop-color="#7A5812"/>
    </linearGradient>
  </defs>
  <!-- Cuerpo principal -->
  <rect x="22" y="18" width="28" height="300" rx="14" fill="url(#body-grad)" stroke="#3A2005" stroke-width="2.2"/>
  <!-- Vetas de bambú -->
  <line x1="26" y1="40" x2="26" y2="306" stroke="#7A5210" stroke-width="0.8" opacity="0.5"/>
  <line x1="46" y1="40" x2="46" y2="306" stroke="#7A5210" stroke-width="0.8" opacity="0.5"/>
  <!-- Nodos del bambú -->
  <rect x="22" y="110" width="28" height="5" rx="2.5" fill="#4A2E08" opacity="0.85"/>
  <rect x="22" y="198" width="28" height="5" rx="2.5" fill="#4A2E08" opacity="0.85"/>
  <rect x="22" y="278" width="28" height="5" rx="2.5" fill="#4A2E08" opacity="0.85"/>
  <!-- Embocadura superior -->
  <rect x="24" y="8" width="24" height="16" rx="6" fill="#2A1403" stroke="#1A0A00" stroke-width="1.5"/>
  <ellipse cx="36" cy="16" rx="7" ry="3.5" fill="#0d0502"/>
  <!-- Tapa inferior -->
  <ellipse cx="36" cy="313" rx="14" ry="5" fill="#4A2E08" stroke="#2A1403" stroke-width="1.5"/>
  <!-- Orificios frontales h1–h6: color inicial = aguamarina (tapado) -->
  <circle id="ph-1" cx="36" cy="128" r="9" fill="#00d4ff" stroke="#007fa8" stroke-width="1.5"/>
  <circle id="ph-2" cx="36" cy="158" r="9" fill="#00d4ff" stroke="#007fa8" stroke-width="1.5"/>
  <circle id="ph-3" cx="36" cy="188" r="9" fill="#00d4ff" stroke="#007fa8" stroke-width="1.5"/>
  <circle id="ph-4" cx="36" cy="216" r="9" fill="#00d4ff" stroke="#007fa8" stroke-width="1.5"/>
  <circle id="ph-5" cx="36" cy="244" r="9" fill="#00d4ff" stroke="#007fa8" stroke-width="1.5"/>
  <circle id="ph-6" cx="36" cy="270" r="9" fill="#00d4ff" stroke="#007fa8" stroke-width="1.5"/>
  <!-- Etiquetas -->
  <text x="54" y="131" font-size="8" fill="#555" font-family="Arial">Si</text>
  <text x="54" y="161" font-size="8" fill="#555" font-family="Arial">La</text>
  <text x="54" y="191" font-size="8" fill="#555" font-family="Arial">Sol</text>
  <text x="54" y="219" font-size="8" fill="#555" font-family="Arial">Fa#</text>
  <text x="54" y="247" font-size="8" fill="#555" font-family="Arial">Mi</text>
  <text x="54" y="273" font-size="8" fill="#555" font-family="Arial">Re</text>
</svg>"""

    html = f"""
<style>
#nota-panel {{
  font-family: 'Courier New', monospace;
  background: #0d1117;
  border-radius: 14px;
  padding: 18px 20px;
  color: #e6edf3;
  user-select: none;
  max-width: 720px;
}}
#panel-top {{
  display: flex;
  gap: 22px;
  align-items: center;
}}
#pincuyo-wrap {{
  display: flex;
  flex-direction: column;
  align-items: center;
}}
#pincuyo-label {{
  font-size: 10px;
  color: #555;
  margin-top: 6px;
  font-family: Arial;
  letter-spacing: 1px;
  text-align: center;
}}
#nota-info {{ flex: 1; }}
#nota-central {{
  font-size: 76px;
  font-weight: bold;
  color: #00d4ff;
  text-shadow: 0 0 24px #00d4ff66;
  min-height: 90px;
  line-height: 90px;
  letter-spacing: 4px;
  transition: color 0.12s, text-shadow 0.12s;
}}
#nota-freq {{
  font-size: 16px;
  color: #8b949e;
  margin-top: 2px;
  min-height: 22px;
}}
#nota-contexto {{
  display: flex;
  align-items: center;
  gap: 24px;
  margin-top: 10px;
  color: #444;
}}
#nota-contexto .lbl {{ font-size: 11px; color: #555; display:block; text-align:center; }}
#nota-ant, #nota-sig {{ font-size: 24px; font-weight: bold; min-width: 50px; text-align: center; }}
#nota-ant {{ color: #58a6ff88; }}
#nota-sig {{ color: #3fb95088; }}
#dig-leyenda {{
  margin-top: 10px;
  font-size: 11px;
  color: #555;
  display: flex;
  gap: 14px;
  align-items: center;
}}
#dig-leyenda span {{ display:flex; align-items:center; gap:5px; }}
.leg-circle {{
  display: inline-block;
  width: 12px; height: 12px;
  border-radius: 50%;
  border: 1.5px solid #444;
}}
#progreso-notas {{
  display: flex;
  flex-wrap: wrap;
  gap: 4px;
  margin-top: 14px;
  max-height: 72px;
  overflow-y: auto;
  padding: 4px;
  background: #161b22;
  border-radius: 8px;
}}
.nota-pill {{
  padding: 2px 8px;
  border-radius: 8px;
  font-size: 11px;
  background: #21262d;
  color: #8b949e;
  transition: background 0.08s, color 0.08s;
  cursor: default;
}}
.nota-pill.pasada  {{ background: #1f6feb44; color: #58a6ff; }}
.nota-pill.activa  {{ background: #00d4ff; color: #0d1117; font-weight: bold; box-shadow: 0 0 8px #00d4ff99; }}
#audio-player {{ width: 100%; margin-top: 14px; }}
</style>
<div id="nota-panel">
  <div id="panel-top">
    <div id="pincuyo-wrap">
      {pincuyo_svg}
      <div id="pincuyo-label">pincuyo<br>6 orificios</div>
      <div id="dig-leyenda">
        <span><span class="leg-circle" style="background:#00d4ff; box-shadow:0 0 6px #00d4ff99;"></span>tapado</span>
        <span><span class="leg-circle" style="background:#0d0605; border-color:#555;"></span>abierto</span>
      </div>
    </div>
    <div id="nota-info">
      <div id="nota-central">▶ Play</div>
      <div id="nota-freq"></div>
      <div id="nota-contexto">
        <div><span class="lbl">anterior</span><span id="nota-ant">—</span></div>
        <span style="color:#333; font-size:28px;">|</span>
        <div><span class="lbl">siguiente</span><span id="nota-sig">—</span></div>
      </div>
    </div>
  </div>
  <div id="progreso-notas">{pills_html}</div>
  <audio id="audio-player" controls>
    <source src="data:audio/wav;base64,{audio_b64}" type="audio/wav">
  </audio>
</div>
<script>
(function() {{
  const notes   = {json.dumps(notes_js)};
  const player  = document.getElementById('audio-player');
  const central = document.getElementById('nota-central');
  const freqEl  = document.getElementById('nota-freq');
  const antEl   = document.getElementById('nota-ant');
  const sigEl   = document.getElementById('nota-sig');
  const pills   = notes.map((_, i) => document.getElementById('pill-' + i));
  const holeIds = ['ph-1','ph-2','ph-3','ph-4','ph-5','ph-6'];
  let lastIdx   = -2;

  const C_CLOSED = '#00d4ff';   // aguamarina = tapado
  const S_CLOSED = '#007fa8';
  const C_OPEN   = '#0d0605';   // negro = abierto
  const S_OPEN   = '#555555';

  function updatePincuyo(dig) {{
    holeIds.forEach(function(id, i) {{
      const el = document.getElementById(id);
      if (!el) return;
      if (dig[i] === 1) {{
        el.setAttribute('fill', C_CLOSED);
        el.setAttribute('stroke', S_CLOSED);
      }} else {{
        el.setAttribute('fill', C_OPEN);
        el.setAttribute('stroke', S_OPEN);
      }}
    }});
  }}

  updatePincuyo([1,1,1,1,1,1]);  // estado inicial: todos tapados (azul)

  player.addEventListener('timeupdate', function() {{
    const t = player.currentTime;
    let idx = -1;
    for (let i = 0; i < notes.length; i++) {{
      if (t >= notes[i][0] && t < notes[i][1]) {{ idx = i; break; }}
    }}
    if (idx === lastIdx) return;
    lastIdx = idx;

    pills.forEach(function(p, i) {{
      p.className = 'nota-pill' + (i < idx ? ' pasada' : (i === idx ? ' activa' : ''));
    }});
    if (idx !== -1 && pills[idx]) pills[idx].scrollIntoView({{ block: 'nearest', behavior: 'smooth' }});

    if (idx !== -1) {{
      central.textContent = notes[idx][3];
      central.style.color = '#00d4ff';
      central.style.textShadow = '0 0 24px #00d4ff66';
      freqEl.textContent  = notes[idx][4].toFixed(1) + ' Hz';
      antEl.textContent   = idx > 0 ? notes[idx-1][3] : '—';
      sigEl.textContent   = idx < notes.length-1 ? notes[idx+1][3] : '—';
      updatePincuyo(notes[idx][5]);
    }} else {{
      central.textContent = '— — —';
      central.style.color = '#333';
      central.style.textShadow = 'none';
      freqEl.textContent  = '';
      antEl.textContent   = '—';
      sigEl.textContent   = '—';
      updatePincuyo([1,1,1,1,1,1]);
    }}
  }});

  player.addEventListener('ended', function() {{
    central.textContent = '✓';
    central.style.color = '#3fb950';
    central.style.textShadow = 'none';
    freqEl.textContent  = 'Fin de la canción';
    updatePincuyo([0,0,0,0,0,0]);
    lastIdx = -2;
  }});
}})();
</script>
"""
    print(f"▶ {len(note_events)} notas — {total_dur:.1f} s")
    display(HTML(html))
else:
    print("⚠ No hay notas para reproducir.")


▶ 104 notas — 41.3 s
